# 13 — Reliability-qualified NTL–load alignment

Evaluate native-daily NTL–NGCP alignment in Samar–Leyte across eight cumulative GHSL masks and spatial-completeness thresholds. Reproduce the daily component heatmaps and the full-record/Haiyan alignment panels.

## Key takeaways

Compare Pearson r with direct-date coverage and paired days. The daily score is (N%/100) × max(r, 0)². The Figure 7 plotting workflow retains its supplied preprocessing, which differs from the capped daily sweep and is stated beside the plot.

## Run order and exports

Run the cells in order from the repository or notebooks/ folder. Figures use outputs/figures/main/, supplementary/ or diagnostics/. Tables use the matching outputs/tables/ folders. Every static export writes HTML, SVG, PNG and JPEG from the same layout. HTML includes Plotly for offline viewing. JPEG preserves the PNG's pixel dimensions and composites transparency on white.

In [ ]:
from io import BytesIO
from PIL import Image as PILImage

from pathlib import Path
import glob
import os
import json

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr

def export_figure(
    fig, output_path, width=None, height=None, scale=3,
    export_html=True, force_date_axis=False, tickformat="%Y", static=True,
):
    """Export one figure layout as HTML, SVG, PNG and JPEG."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    figure = go.Figure(json.loads(pio.to_json(fig, validate=True)))
    figure.update_layout(
        width=width if width is not None else figure.layout.width or 1200,
        height=height if height is not None else figure.layout.height or 600,
    )
    if force_date_axis:
        figure.update_xaxes(type="date", tickformat=tickformat)

    if static:
        png_bytes = figure.to_image(format="png", scale=scale)
        svg_bytes = figure.to_image(format="svg", scale=1)
        with PILImage.open(BytesIO(png_bytes)) as image:
            rgba = image.convert("RGBA")
            jpeg = PILImage.new("RGB", rgba.size, "white")
            jpeg.paste(rgba, mask=rgba.getchannel("A"))
        buffer = BytesIO()
        jpeg.save(buffer, format="JPEG", quality=95, subsampling=0)
        output_path.with_suffix(".png").write_bytes(png_bytes)
        output_path.with_suffix(".svg").write_bytes(svg_bytes)
        output_path.with_suffix(".jpeg").write_bytes(buffer.getvalue())
    if export_html:
        figure.write_html(
            str(output_path.with_suffix(".html")),
            include_plotlyjs=True, full_html=True,
        )
    return fig

### 1.2 Project root, input paths, and output folders

This notebook uses the standard repository output structure.

| Output type | Folder |
|---|---|
| Main manuscript figures | `outputs/revisions/figures/main/` |
| Supplementary figures | `outputs/revisions/figures/supplementary/` |
| Main manuscript tables | `outputs/tables/main/` |
| Supplementary tables | `outputs/tables/supplementary/` |

In [ ]:
working_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        directory
        for directory in (working_dir, *working_dir.parents)
        if (directory / "notebooks").is_dir() and (directory / "datasets").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Open the kernel from the repository or its notebooks folder. "
        "The repository must contain sibling notebooks/ and datasets/ folders."
    )

NTL_GHSL_DIR = (
    PROJECT_ROOT
    / "datasets"
    / "vnp46a2"
    / "ghsl_masked_ntl_samar_leyte"
)

NGCP_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "ngcp"
    / "NGCP_Hourly_Demand.csv"
)

OUTPUT_MAIN_FIG_DIR = PROJECT_ROOT / "outputs" / "figures" / "main"
OUTPUT_SUPP_FIG_DIR = PROJECT_ROOT / "outputs" / "figures" / "supplementary"
OUTPUT_MAIN_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables" / "main"
OUTPUT_SUPP_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables" / "supplementary"

for out_dir in [
    OUTPUT_MAIN_FIG_DIR,
    OUTPUT_SUPP_FIG_DIR,
    OUTPUT_MAIN_TABLE_DIR,
    OUTPUT_SUPP_TABLE_DIR,
]:
    out_dir.mkdir(parents=True, exist_ok=True)

print(f"NTL GHSL directory: {NTL_GHSL_DIR.relative_to(PROJECT_ROOT)}")
print(f"NGCP file: {NGCP_PATH.relative_to(PROJECT_ROOT)}")

### 1.4 Analysis settings

In [ ]:
# ==============================
# Analysis settings
# ==============================

START_DATE = pd.Timestamp("2012-01-19")
END_DATE = pd.Timestamp("2024-12-31")
TOTAL_DAYS = (END_DATE - START_DATE).days + 1

VALUE_COL = "Mean_by_sum"
ROLLING_WINDOW = 30
CENTERED_MA = True

THRESHOLDS_MAIN = list(range(5, 100, 5))
THRESHOLDS_DISPLAY = [10, 25, 50, 75, 90]

STORM_EVENTS = [
    {"name": "Haiyan", "date": "2013-11-08"},
    {"name": "Rai", "date": "2021-12-16"},
]

## 2. GHSL Thematic Masks

### 2.1 Cumulative GHSL mask definitions

The GHSL masks are cumulative settlement supports. They expand from the most intensely urban class to broader settlement and land classes.

| Code | GHSL range | Label |
|---|---|---|
| G1 | 30 | Urban Core |
| G2 | 23–30 | Dense Urban |
| G3 | 22–30 | Core & Inner |
| G4 | 21–30 | Urban Fringe |
| G5 | 13–30 | Built-Up / Rural Mix |
| G6 | 12–30 | Low–Mid Rural |
| G7 | 11–30 | Rural Settlements |
| G8 | 10–30 | All Land |

This design tests whether electricity-load alignment is strongest in dense, consistently illuminated urban supports and weakens as lower-density or dimmer classes are added.


In [ ]:
# ==============================
# GHSL thematic mask metadata
# ==============================

GHSL_META = {
    "30":    {"code": "G1", "label": "Urban Core",            "color": "#ff0000"},
    "23-30": {"code": "G2", "label": "Dense Urban",           "color": "#732600"},
    "22-30": {"code": "G3", "label": "Core & Inner",          "color": "#a87000"},
    "21-30": {"code": "G4", "label": "Urban Fringe",          "color": "#ffff00"},
    "13-30": {"code": "G5", "label": "Built-Up / Rural Mix",  "color": "#375623"},
    "12-30": {"code": "G6", "label": "Low–Mid Rural",         "color": "#abcd66"},
    "11-30": {"code": "G7", "label": "Rural Settlements",     "color": "#cdf57a"},
    "10-30": {"code": "G8", "label": "All Land",              "color": "#7ab6f5"},
}

GHSL_ORDER = ["30", "23-30", "22-30", "21-30", "13-30", "12-30", "11-30", "10-30"]

ghsl_mask_table = pd.DataFrame([
    {
        "Mask": GHSL_META[g]["code"],
        "GHSL Range": g,
        "Label": GHSL_META[g]["label"],
    }
    for g in GHSL_ORDER
])

ghsl_mask_table

In [ ]:
ghsl_mask_table.to_csv(
    OUTPUT_SUPP_TABLE_DIR / "table_sx_ghsl_thematic_masks_samar_leyte.csv",
    index=False,
)

## 3. Load GHSL-Masked NTL and NGCP Load Data

### 3.1 Load GHSL-masked daily NTL statistics

Each CSV represents one cumulative GHSL thematic mask. The files are combined into a single long-format panel with a `Group` column.

Expected NTL columns:

| Column | Description |
|---|---|
| `date` or `system:index` | Observation date |
| `Group` | GHSL thematic mask range |
| `Mean_by_sum` | Mask-mean DNB-BRDF radiance |
| `NTL_mean` | Alternative daily mean radiance |
| `NTL_std` | Daily within-mask standard deviation |
| `Valid_px` | Number of valid observed pixels |

In [ ]:
csv_files = sorted(glob.glob(str(NTL_GHSL_DIR / "*.csv")))

if len(csv_files) == 0:
    raise FileNotFoundError(f"No CSV files found in: {NTL_GHSL_DIR.relative_to(PROJECT_ROOT)}")

df_list = []

for file in csv_files:
    file_path = Path(file)

    group_name = (
        file_path.name
        .replace("EasternVisayas_DNBBRDF_", "")
        .replace("_stats.csv", "")
    )

    temp = pd.read_csv(file_path)

    if "date" not in temp.columns:
        if "system:index" in temp.columns:
            temp["date"] = (
                temp["system:index"]
                .astype(str)
                .str.extract(r"(\d{4}_\d{2}_\d{2})")[0]
                .str.replace("_", "-", regex=False)
            )
        else:
            raise ValueError(f"No date or system:index column found in {file_path.name}")

    temp["date"] = pd.to_datetime(temp["date"], errors="coerce")
    temp["Group"] = group_name

    temp = temp[
        (temp["date"] >= START_DATE)
        & (temp["date"] <= END_DATE)
    ].copy()

    df_list.append(temp)

df_all = pd.concat(df_list, ignore_index=True)
df_all["Group"] = df_all["Group"].astype(str)

df_all = df_all.sort_values(["Group", "date"]).reset_index(drop=True)

print(f"Rows: {len(df_all):,}")
print(f"Groups: {sorted(df_all['Group'].unique())}")
print(f"Date range: {df_all['date'].min().date()} to {df_all['date'].max().date()}")

df_all.head()

In [ ]:
df_all.to_csv(
    OUTPUT_SUPP_TABLE_DIR / "table_sx_samar_leyte_ghsl_masked_ntl_panel.csv",
    index=False,
)

### 3.2 Load NGCP 1 AM electricity load

NGCP hourly load is reduced to the 1 AM value to approximate the VIIRS nighttime overpass window.

Expected NGCP columns after cleaning:

| Column | Description |
|---|---|
| `Date` | Daily date |
| `Grid_1AM_MW` | NGCP 1 AM load in megawatts |

In [ ]:
# ==============================
# Load NGCP hourly demand and extract 1 AM load
# ==============================

df_ngcp_raw = pd.read_csv(NGCP_PATH, skiprows=2)

df_ngcp_raw.columns = ["Date", "Coincident Peak", "Hour No"] + [str(i) for i in range(1, 25)]

df_ngcp = (
    df_ngcp_raw[["Date", "1"]]
    .rename(columns={"1": "Grid_1AM_MW"})
    .copy()
)

df_ngcp["Date"] = pd.to_datetime(df_ngcp["Date"], errors="coerce", dayfirst=True)
df_ngcp["Grid_1AM_MW"] = pd.to_numeric(df_ngcp["Grid_1AM_MW"], errors="coerce")

df_ngcp = df_ngcp[
    (df_ngcp["Date"] >= START_DATE)
    & (df_ngcp["Date"] <= END_DATE)
].copy()

df_ngcp = df_ngcp.sort_values("Date").dropna(subset=["Date", "Grid_1AM_MW"])

print(f"Rows: {len(df_ngcp):,}")
print(f"Date range: {df_ngcp['Date'].min().date()} to {df_ngcp['Date'].max().date()}")

df_ngcp.head()

In [ ]:
df_ngcp.to_csv(
    OUTPUT_SUPP_TABLE_DIR / "table_sx_ngcp_1am_load_samar_leyte_2012_2024.csv",
    index=False,
)

## 4. Shared Preprocessing Functions

### 4.1 Reliability-qualified NTL preprocessing

The alignment workflow uses the same preprocessing sequence for every GHSL mask and threshold:

1. Remove days with no valid observed pixels.
2. Compute spatial completeness, `SC (%)`.
3. Winsorise the NTL series at the 0–95th percentile.
4. Retain days satisfying `SC ≥ τ`.
5. Apply a 30-day moving average.
6. Align with NGCP load on overlapping dates.
7. Standardise both series using z-scores.

This keeps the comparison scale-neutral and makes the metrics reflect co-movement rather than differences in units or class brightness.

In [ ]:
# ==============================
# Event Markers
# ==============================

STORM_MARKERS = [
    ("Haiyan", "2013-11-08"),
    ("Rai", "2021-12-16"),
]


def add_event_markers(fig, y=1.04):
    for name, event_date in STORM_MARKERS:
        fig.add_vline(
            x=event_date,
            line_dash="dash",
            line_color="black",
            line_width=2,
        )

        fig.add_annotation(
            x=event_date,
            y=y,
            yref="paper",
            text=name,
            showarrow=False,
            font=dict(size=14, color="black"),
            bgcolor="white",
            bordercolor="black",
            borderwidth=1,
            xanchor="left",
        )

    return fig


def hex_to_rgba(hex_color, alpha=0.5):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"


def prepare_ntl_group(
    gdf,
    tau=10,
    value_col=VALUE_COL,
    p_lo=0,
    p_hi=95,
    w=ROLLING_WINDOW,
    centered=CENTERED_MA,
):
    ntl = gdf.copy()
    ntl = ntl[ntl["Valid_px"] > 0].copy()
    ntl["date"] = pd.to_datetime(ntl["date"])
    ntl = ntl.sort_values("date")

    if ntl.empty:
        return ntl

    total_px = ntl["Valid_px"].max()
    ntl["SC_pct"] = ntl["Valid_px"] / total_px * 100.0

    lo = np.nanpercentile(ntl[value_col], p_lo)
    hi = np.nanpercentile(ntl[value_col], p_hi)

    ntl["_val"] = np.clip(ntl[value_col], lo, hi)

    ntl = ntl[ntl["SC_pct"] >= tau].copy()

    if ntl.empty:
        return ntl

    ntl["NTL_MA"] = (
        ntl["_val"]
        .rolling(w, min_periods=1, center=centered)
        .mean()
    )

    return ntl


def prepare_ngcp_load(
    df_ngcp,
    w=ROLLING_WINDOW,
    centered=CENTERED_MA,
):
    ng = df_ngcp.copy()
    ng["Date"] = pd.to_datetime(ng["Date"])
    ng = ng.sort_values("Date")

    ng["Load_MA"] = (
        ng["Grid_1AM_MW"]
        .rolling(w, min_periods=1, center=centered)
        .mean()
    )

    return ng


def zscore(series):
    series = pd.Series(series).astype(float)
    std = series.std(ddof=0)

    if std == 0 or not np.isfinite(std):
        return pd.Series(np.nan, index=series.index)

    return (series - series.mean()) / std


## 5. Settlement-Aggregated NTL with Reliability Filter

### 5.1 Interactive diagnostic: SC filtering and smoothed NTL

This diagnostic visualises how spatial completeness filtering changes the GHSL-aggregated DNB-BRDF time series. The upper panel shows daily spatial completeness, while the lower panel shows daily radiance and the 30-day moving average after threshold filtering.

This figure is primarily exploratory and is exported as HTML only.

In [ ]:
def plot_timeseries_with_buttons(df_all, threshold=40, w=ROLLING_WINDOW):
    groups = [g for g in GHSL_ORDER if g in df_all["Group"].unique()]

    fig = make_subplots(
        rows=2,
        cols=1,
        row_heights=[0.15, 0.75],
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=("Daily spatial completeness, SC (%)", "Reliability-qualified mean DNB-BRDF"),
    )

    buttons = []
    n_traces_per_group = 3

    for i, group in enumerate(groups):
        gdf = df_all[df_all["Group"] == group].copy()
        base = prepare_ntl_group(gdf, tau=0, w=w)

        if base.empty:
            continue

        plot_df = prepare_ntl_group(gdf, tau=threshold, w=w)

        code = GHSL_META[group]["code"]
        label = GHSL_META[group]["label"]
        color = GHSL_META[group]["color"]

        fig.add_trace(
            go.Heatmap(
                z=np.expand_dims(base["SC_pct"].values, axis=0),
                x=base["date"].dt.strftime("%Y-%m-%d"),
                y=[0],
                colorscale="Greens",
                zmin=0,
                zmax=100,
                showscale=True,
                colorbar=dict(title="SC (%)", len=0.22, thickness=16, x=0.96, y=0.95),
                visible=(i == 0),
            ),
            row=1,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=plot_df["date"].dt.strftime("%Y-%m-%d"),
                y=plot_df["_val"],
                mode="markers",
                marker=dict(color=color, size=4, opacity=0.18),
                name=f"{code} daily",
                visible=(i == 0),
                showlegend=False,
            ),
            row=2,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=plot_df["date"].dt.strftime("%Y-%m-%d"),
                y=plot_df["NTL_MA"],
                mode="lines",
                line=dict(color=color, width=2.2),
                name=f"{code}: {label}",
                visible=(i == 0),
            ),
            row=2,
            col=1,
        )

        visibility = [False] * (n_traces_per_group * len(groups))
        for j in range(n_traces_per_group):
            visibility[i * n_traces_per_group + j] = True

        buttons.append(
            dict(
                label=code,
                method="update",
                args=[
                    {"visible": visibility},
                    {"title": f"Samar–Leyte DNB-BRDF NTL — {code} (SC ≥ {threshold}%)"},
                ],
            )
        )

    add_event_markers(fig, y=0.86)

    fig.update_layout(
        updatemenus=[
            dict(
                type="buttons",
                buttons=buttons,
                direction="right",
                showactive=True,
                x=0.5,
                xanchor="center",
                y=-0.4,
                yanchor="top",
            )
        ],
        xaxis2=dict(title="Date", rangeslider=dict(visible=True), type="date"),
        yaxis=dict(showticklabels=False),
        yaxis2=dict(title="Radiance (nW·cm⁻²·sr⁻¹)"),
        template="plotly_white",
        paper_bgcolor="rgba(0,0,0,0)",
        legend=dict(orientation="h", y=0.85, x=0.78, font=dict(size=14)),
        height=650,
        width=1200,
        margin=dict(t=70, l=70, b=110, r=40),
    )

    return fig


fig_sx_interactive_ntl = plot_timeseries_with_buttons(df_all, threshold=40, w=ROLLING_WINDOW)
fig_sx_interactive_ntl.show()

In [ ]:
export_figure(fig_sx_interactive_ntl, Path(OUTPUT_SUPP_FIG_DIR / "fig_sx_interactive_ntl_reliability_filter_samar_leyte.html").with_suffix(""), static=True)

### 5.2 Diagnostic discussion

The reliability-filtered NTL time series shows that spatial completeness is highly intermittent across the full record. Filtering removes the weakest observation days, while the 30-day moving average preserves broad structural changes, including the post-Haiyan disruption and recovery trajectory.

This diagnostic motivates the formal threshold sweep below. The key issue is not whether a daily VNP46A2 layer exists, but whether enough of the target settlement support was directly observed to justify interpreting that day.


## 6. Native-Daily and 30-Day NTL–Load Alignment

### 6.1 Samar–Leyte: full record and Haiyan window

The main alignment figures use the regional cumulative GHSL-mask exports in
`datasets/vnp46a2/visayas_ghsl_masked_daily_stats/` and 01:00 electricity load from
`datasets/ngcp/NGCP_hourly_load.xlsx`. Each mask occupies a separate row. Native-daily
observations (W=1) are shown as faint points, and 30-day trailing calendar means
(W=30) as thin lines. Missing observations remain missing; a moving mean is
available when at least one observation occurs within its calendar window.

NTL observations above the within-mask 95th percentile are excluded before
observed-maximum support normalisation and the strict SC > τ filter. NGCP outliers
are screened using a centred 31-day median and median absolute deviation, with a
three-sigma cutoff and at least seven available observations. NTL z-scores use the
full-record daily paired observations as their reference; NGCP z-scores use its
screened full-record observations. Both full-record and event panels retain these
normalisations. The annotations report panel-specific r₁, r₃₀, native-daily paired
coverage (N%) and paired-day count (n).

The Haiyan window spans 60 days before to 180 days after 8 November 2013.

## 6. Spatial completeness and signal structure

Inspect the retained supplementary time series, spatial-completeness distributions and retention curves. These show how a change in the threshold alters observable support through time. The Figure S8 series is from Notebook 03. Notebook 23 contains another layout whose source-preparation helper was absent from the provided notebooks.

In [ ]:
# ==============================
# Static raw-radiance NTL–NGCP threshold figure
# ==============================

def plot_raw_alignment_static(
    df_all,
    df_ngcp,
    tau=10,
    w=ROLLING_WINDOW,
    value_col=VALUE_COL,
):
    groups = [g for g in GHSL_ORDER if g in df_all["Group"].unique()]
    ng = prepare_ngcp_load(df_ngcp, w=w)

    fig = go.Figure()

    for group in groups:
        gdf = df_all[df_all["Group"] == group].copy()
        ntl = prepare_ntl_group(gdf, tau=tau, value_col=value_col, w=w)

        if ntl.empty:
            continue

        color = GHSL_META[group]["color"]
        code = GHSL_META[group]["code"]
        label = GHSL_META[group]["label"]

        fig.add_trace(
            go.Scatter(
                x=ntl["date"].dt.strftime("%Y-%m-%d"),
                y=ntl["NTL_MA"],
                mode="lines",
                line=dict(color=color, width=2),
                name=f"{code}: {label}",
                yaxis="y1",
            )
        )

    fig.add_trace(
        go.Scatter(
            x=ng["Date"].dt.strftime("%Y-%m-%d"),
            y=ng["Load_MA"],
            mode="lines",
            line=dict(color="black", width=3),
            name="NGCP load",
            yaxis="y2",
        )
    )

    add_event_markers(fig, y=1.04)

    fig.update_layout(
        template="plotly_white",
        width=1200,
        height=620,
        margin=dict(t=70, b=70, l=70, r=270),
        paper_bgcolor="rgba(0,0,0,0)",
        xaxis=dict(title="Date", type="date"),
        yaxis=dict(title="Radiance (nW·cm⁻²·sr⁻¹)", side="left"),
        yaxis2=dict(
            title="Load (MW)",
            overlaying="y",
            side="right",
            showgrid=False,
        ),
        legend=dict(
            orientation="v",
            x=1.08,
            y=1,
            font=dict(size=13),
        ),
        title=f"GHSL-stratified DNB-BRDF and NGCP load, τ = {tau}%",
    )

    return fig
# ==============================
# Export Supplementary Figure S8 panels
# ==============================

fig_s8a = plot_raw_alignment_static(df_all, df_ngcp, tau=10)
fig_s8b = plot_raw_alignment_static(df_all, df_ngcp, tau=50)
fig_s8c = plot_raw_alignment_static(df_all, df_ngcp, tau=90)

fig_s8a.show()
fig_s8b.show()
fig_s8c.show()

In [ ]:
export_figure(
    fig_s8a,
    OUTPUT_SUPP_FIG_DIR / "fig_s8a_raw_ntl_ngcp_tau10",
    width=1200,
    height=620,
    scale=3,
    force_date_axis=True,
    tickformat="%Y",
)

export_figure(
    fig_s8b,
    OUTPUT_SUPP_FIG_DIR / "fig_s8b_raw_ntl_ngcp_tau50",
    width=1200,
    height=620,
    scale=3,
    force_date_axis=True,
    tickformat="%Y",
)

export_figure(
    fig_s8c,
    OUTPUT_SUPP_FIG_DIR / "fig_s8c_raw_ntl_ngcp_tau90",
    width=1200,
    height=620,
    scale=3,
    force_date_axis=True,
    tickformat="%Y",
)


### 7.2 Supplementary figure discussion

At low thresholds, more dates are retained and the full temporal trajectory remains visible, but weakly supported observations can contribute more strongly. At intermediate thresholds, the broad temporal pattern remains interpretable while lower-observability days are removed. At very high thresholds, the retained series becomes fragmented because the spatial-support requirement removes most daily observations.

This comparison makes the reliability–coverage trade-off visible. Increasing `τ` does not simply improve the time series; it also changes continuity, amplitude, and representation of the underlying system dynamics.

In [ ]:
def plot_sc_distribution_profiles(
    df_all,
    thresholds=[25, 50, 75, 90],
    bin_size=10,
    value_col=VALUE_COL,
):
    groups = [g for g in GHSL_ORDER if g in df_all["Group"].unique()]
    bins = np.arange(0, 101, bin_size)
    bin_centers = (bins[:-1] + bins[1:]) / 2

    fig = go.Figure()

    for group in groups:
        gdf = prepare_ntl_group(
            df_all[df_all["Group"] == group].copy(),
            tau=0,
            value_col=value_col,
        )

        if gdf.empty:
            continue

        vals = gdf["SC_pct"].dropna()

        counts, _ = np.histogram(vals, bins=bins)
        percents = counts / counts.sum() * 100 if counts.sum() > 0 else np.zeros_like(counts)

        fig.add_trace(
            go.Scatter(
                x=bin_centers,
                y=percents,
                mode="lines+markers",
                name=f"{GHSL_META[group]['code']}: {GHSL_META[group]['label']}",
                line=dict(color=GHSL_META[group]["color"], width=3),
                marker=dict(size=7, color=GHSL_META[group]["color"]),
            )
        )

    for tau in thresholds:
        fig.add_vline(x=tau, line_dash="dash", line_color="black", line_width=1.5)

    fig.update_layout(
        template="plotly_white",
        xaxis=dict(
            title="Spatial completeness, SC (%)",
            range=[0, 100],
        ),
        yaxis=dict(
            title="Percent of days",
            range=[0, 30],
        ),
        legend=dict(
            orientation="v",
            x=1.02,
            y=1,
            font=dict(size=13),
        ),
        width=1200,
        height=550,
        margin=dict(t=50, l=70, b=70, r=280),
        paper_bgcolor="rgba(0,0,0,0)",
    )

    return fig


fig_sx_sc_dist = plot_sc_distribution_profiles(df_all)
fig_sx_sc_dist.show()

In [ ]:
export_figure(
    fig_sx_sc_dist,
    OUTPUT_SUPP_FIG_DIR / "fig_sx_sc_distribution_profiles_samar_leyte",
    width=1200,
    height=550,
    scale=3,
)

### 11.2 Cumulative retention curves

This figure shows the percentage of days retained as the spatial completeness threshold increases.


In [ ]:
# ==============================
# Cumulative retention curves
# ==============================

def plot_sc_retention_curves(
    df_all,
    thresholds=[25, 50, 75, 90],
    value_col=VALUE_COL,
):
    groups = [g for g in GHSL_ORDER if g in df_all["Group"].unique()]
    tau_range = np.arange(0, 101, 1)

    fig = go.Figure()

    for group in groups:
        gdf = prepare_ntl_group(
            df_all[df_all["Group"] == group].copy(),
            tau=0,
            value_col=value_col,
        )

        if gdf.empty:
            continue

        vals = gdf["SC_pct"].dropna().values
        retained = [(vals >= tau).sum() / len(vals) * 100 for tau in tau_range]

        fig.add_trace(
            go.Scatter(
                x=tau_range,
                y=retained,
                mode="lines",
                name=f"{GHSL_META[group]['code']}: {GHSL_META[group]['label']}",
                line=dict(color=GHSL_META[group]["color"], width=3),
                hovertemplate="τ: %{x:.0f}%<br>Retained days: %{y:.1f}%<extra></extra>",
            )
        )

    for tau in thresholds:
        fig.add_vline(x=tau, line_dash="dash", line_color="black", line_width=1.5)

    fig.update_layout(
        template="plotly_white",
        xaxis=dict(
            title="Spatial completeness threshold, τ (%)",
            range=[0, 100],
            dtick=10,
        ),
        yaxis=dict(
            title="Days retained (%)",
            range=[0, 105],
        ),
        legend=dict(
            orientation="v",
            x=1.02,
            y=1,
            font=dict(size=13),
        ),
        width=1200,
        height=600,
        margin=dict(t=50, l=70, b=70, r=280),
        paper_bgcolor="rgba(0,0,0,0)",
    )

    return fig


fig_sx_retention = plot_sc_retention_curves(df_all)
fig_sx_retention.show()

In [ ]:
export_figure(
    fig_sx_retention,
    OUTPUT_SUPP_FIG_DIR / "fig_sx_sc_retention_curves_samar_leyte",
    width=1200,
    height=600,
    scale=3,
)


### 11.3 Supplementary discussion

The SC distribution and retention curves show why threshold choice must be treated as a tunable reliability parameter. Coverage declines monotonically as `τ` increases, but the rate of decline differs among settlement supports.

Urban-focused masks retain more high-completeness days, while broader rural-dominant masks lose continuity more quickly. This explains why stricter thresholds can improve statistical alignment for some configurations while also reducing temporal representation.


## 7. Native-daily mask and threshold selection

The exact search uses 1 percentage-point increments and the Figure 6 panels show the 5% display grid. At $W=1$, $N\%$ is the share of load-available dates with a qualified NTL–NGCP pair. The revised score is $S=(N\%/100)\max(r,0)^2$. The Pearson $r=0.60$ and coverage $N=50\%$ colour limits are display references. Cell annotations show the underlying values at the displayed precision. The full Samar–Leyte window sweep also supplies the sensitivity notebook.

In [ ]:
import warnings





from IPython.display import display, Markdown

ROOT = PROJECT_ROOT
OUTPUT_ROOT = PROJECT_ROOT / "outputs"

FIG_MAIN = OUTPUT_ROOT / "figures" / "main"
FIG_SUPP = OUTPUT_ROOT / "figures" / "supplementary"
FIG_DIAGNOSTIC = OUTPUT_ROOT / "figures" / "diagnostics"

TABLE_MAIN = OUTPUT_ROOT / "tables" / "main"
TABLE_SUPP = OUTPUT_ROOT / "tables" / "supplementary"

for folder in (
    FIG_MAIN,
    FIG_SUPP,
    FIG_DIAGNOSTIC,
    TABLE_MAIN,
    TABLE_SUPP,
):
    folder.mkdir(parents=True, exist_ok=True)

# Match Notebook 04's multi-subgrid analysis period and input locations.
DATE_START = pd.Timestamp("2012-01-19")
DATE_END = pd.Timestamp("2024-12-31")
VALIDATION_DATES = pd.date_range(DATE_START, DATE_END, freq="D", name="date")
NTL_FOLDER = ROOT / "datasets" / "vnp46a2" / "visayas_ghsl_masked_daily_stats"
NGCP_XLSX_PATH = ROOT / "datasets" / "ngcp" / "NGCP_hourly_load.xlsx"

WINDOWS = np.arange(1, 31)
WINDOW_CHECKPOINTS = [1, 4, 7, 14, 21, 30]
SC_GRID = np.arange(0, 101, 1, dtype=float)  # exact search and CSV output
MAIN_TAUS = np.arange(0, 101, 5, dtype=float)  # retain the paper's display interval
SUPP_TAUS = MAIN_TAUS.copy()                   # consistent figures throughout
PRIMARY_REGION = "Samar-Leyte"
EVENTS = [
    ("Haiyan", pd.Timestamp("2013-11-08"), "2013-05-01", "2014-05-31"),
    ("Rai", pd.Timestamp("2021-12-16"), "2021-07-01", "2022-07-31"),
]

GROUPS = [
    ("G8", "10-30", "All Land", "#4E79A7"),
    ("G7", "11-30", "Rural Settlements", "#A8CE74"),
    ("G6", "12-30", "Low–Mid Rural", "#7BAF54"),
    ("G5", "13-30", "Built-Up / Rural Mix", "#375623"),
    ("G4", "21-30", "Urban Fringe", "#D9B900"),
    ("G3", "22-30", "Core & Inner", "#E6A23C"),
    ("G2", "23-30", "Dense Urban", "#A87000"),
    ("G1", "30", "Urban Core", "#D65A4A"),
]
GROUP_LOOKUP = {
    code: {"suffix": suffix, "label": label, "color": color}
    for code, suffix, label, color in GROUPS
}
ORDERED_MASKS = [f"G{i}" for i in range(1, 9)]
MASK_COLORS = {code: GROUP_LOOKUP[code]["color"] for code in ORDERED_MASKS}

REGION_RENAME = {
    "CebuProvince": "Cebu",
    "SamarLeyte": "Samar-Leyte",
    "NegrosIsland": "Negros",
    "PanayIsland": "Panay",
    "BoholProvince": "Bohol",
}
REGION_ORDER = ["Samar-Leyte", "Bohol", "Cebu", "Negros", "Panay"]
GROUP_RENAME = {"30-only": "30"}
SHEET_MAP = {
    "Panay": "PANAY HOURLY LOAD 2013-2024",
    "Negros": "NEGROS HOURLY LOAD 2013-2024",
    "Samar-Leyte": "LEY-SAM HOURLY LOAD 2013-2024",
    "Cebu": "CEBU HOURLY LOAD 2013-2024",
    "Bohol": "BOHOL LOAD 2013-2024",
}

PLOT_TEMPLATE = "plotly_white"
INK = "#233B60"
GRID = "#E8EDF3"
R_CONTRAST_SCALE = "jet_r"
N_CONTRAST_SCALE = "Viridis"
SCORE_CONTRAST_SCALE = "Turbo"
DIVERGING_SCALE = [[0.00, "#B2182B"], [0.50, "#F7F7F7"], [1.00, "#2166AC"]]
POSITIVE_SCALE = "Viridis"
R_COLOUR_POWER = 2.0
N_COLOUR_POWER = 3.0
R_TICKS = np.array([-1.0, -0.5, 0.0, 0.5, 0.75, 1.0])
R_COLOUR_TICKS = np.sign(R_TICKS) * np.abs(R_TICKS) ** R_COLOUR_POWER
N_TICKS = np.array([0, 50, 75, 90, 100], dtype=float)
N_COLOUR_TICKS = (N_TICKS / 100) ** N_COLOUR_POWER

In [ ]:
def parse_filename(filepath):
    name = Path(filepath).name.replace("_stats.csv", "")
    parts = name.split("_DNBBRDF_class")
    if len(parts) != 2:
        return "Unknown", "Unknown"
    region, group = parts
    return REGION_RENAME.get(region, region), GROUP_RENAME.get(group, group)


def read_multiregion_ntl(folder):
    csv_files = sorted(Path(folder).glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No NTL CSV files found in {folder.relative_to(PROJECT_ROOT)}")

    frames = []
    for path in csv_files:
        region, group = parse_filename(path)
        if region == "Unknown":
            continue
        try:
            frame = pd.read_csv(path)
        except Exception:
            frame = pd.read_csv(path, dtype=str, engine="python", on_bad_lines="skip")

        if "date" not in frame.columns and "system:index" in frame.columns:
            frame["date"] = (
                frame["system:index"].astype(str)
                .str.extract(r"(\d{4}_\d{2}_\d{2})")[0]
                .str.replace("_", "-", regex=False)
            )
        if "date" not in frame.columns:
            raise ValueError(f"{path.name}: neither date nor system:index is available")

        frame["Region"] = region
        frame["Group"] = group
        frame["source_file"] = path.relative_to(PROJECT_ROOT).as_posix()
        frames.append(frame)

    if not frames:
        raise RuntimeError(f"No recognised multi-region NTL files were loaded from {folder.relative_to(PROJECT_ROOT)}")

    panel = pd.concat(frames, ignore_index=True)
    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize()
    panel = panel[panel["date"].between(DATE_START, DATE_END)].copy()
    panel["Valid_px"] = pd.to_numeric(panel["Valid_px"], errors="coerce")
    panel["Mean_by_sum"] = pd.to_numeric(panel["Mean_by_sum"], errors="coerce")
    return panel.sort_values(["Region", "Group", "date"]).reset_index(drop=True)


def mask_frame(group_frame):
    data = group_frame.copy().drop_duplicates("date").set_index("date").sort_index()
    valid = data["Valid_px"]
    raw = data["Mean_by_sum"].where(valid > 0)
    denominator = valid.loc[raw.notna()].max()
    if not np.isfinite(denominator) or denominator <= 0:
        raise ValueError("No valid direct observations for this region–mask combination")
    p95 = raw.quantile(0.95)
    return pd.DataFrame({
        "raw": raw,
        "ntl": raw.clip(upper=p95),
        "SC": 100 * valid / denominator,
        "Valid_px": valid,
    }).reindex(VALIDATION_DATES)


def load_ngcp_grid_1am_flat(xlsx_path, sheet_map):
    series_by_region = {}
    selected_columns = {}
    plot_series_by_region = {}
    for region, sheet_name in sheet_map.items():
        frame = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=1)
        frame.columns = [str(column).strip() for column in frame.columns]
        date_col = frame.columns[0]
        frame = frame.rename(columns={date_col: "DATE"})
        plot_dates = pd.to_datetime(frame["DATE"], errors="coerce")
        plot_frame = frame.drop(columns="DATE").set_index(plot_dates)
        frame["DATE"] = plot_dates.dt.normalize()
        frame = frame.dropna(subset=["DATE"]).set_index("DATE")
        frame.columns = [str(column).strip() for column in frame.columns]

        possible = ["1", "01", "1.0", "Hour 1", "HOUR 1", "1AM", "1am"]
        hour_col = next((column for column in possible if column in frame.columns), None)
        if hour_col is None:
            numeric_like = [
                column for column in frame.columns
                if str(column).replace(".0", "").isdigit()
            ]
            hour_col = numeric_like[0] if numeric_like else None
        if hour_col is None and len(frame.columns):
            hour_col = frame.columns[0]
        if hour_col is None:
            raise KeyError(f"No usable 01:00 load column in sheet {sheet_name}")

        values = pd.to_numeric(frame[hour_col], errors="coerce")
        values = values.groupby(level=0).mean().reindex(VALIDATION_DATES)
        series_by_region[region] = values.rename(region)
        selected_columns[region] = str(hour_col)
        plot_values = pd.to_numeric(plot_frame[hour_col], errors="coerce")
        plot_series_by_region[region] = plot_values.loc[
            (plot_values.index >= DATE_START) & (plot_values.index <= DATE_END)
        ]

    return pd.DataFrame(series_by_region), selected_columns, pd.DataFrame(plot_series_by_region).sort_index()


NTL_PANEL = read_multiregion_ntl(NTL_FOLDER)
NGCP_PANEL, NGCP_COLUMNS, ngcp_df = load_ngcp_grid_1am_flat(NGCP_XLSX_PATH, SHEET_MAP)

DATA_REGISTRY = {}
registry_rows = []
for region in REGION_ORDER:
    masks = {}
    source_files = []
    for code in ORDERED_MASKS:
        group = GROUP_LOOKUP[code]["suffix"]
        subset = NTL_PANEL[
            NTL_PANEL["Region"].eq(region) & NTL_PANEL["Group"].eq(group)
        ].copy()
        if subset.empty:
            raise FileNotFoundError(
                f"{region}: GHSL group {group} ({code}) is absent from {NTL_FOLDER.relative_to(PROJECT_ROOT)}"
            )
        masks[code] = mask_frame(subset)
        source_files.extend(subset["source_file"].dropna().unique().tolist())

    load = NGCP_PANEL[region].copy()
    DATA_REGISTRY[region] = {"masks": masks, "load": load}
    registry_rows.append({
        "region": region,
        "ntl_files": len(set(source_files)),
        "ghsl_masks": len(masks),
        "ngcp_sheet": SHEET_MAP[region],
        "load_column": NGCP_COLUMNS[region],
        "load_dates": int(load.notna().sum()),
    })

REGISTRY_TABLE = pd.DataFrame(registry_rows)
display(REGISTRY_TABLE)
REGISTRY_TABLE.to_csv(TABLE_SUPP / "data_registry.csv", index=False)

In [ ]:
def paired_metrics(ntl, load, direct_mask, window_days):
    frame = pd.DataFrame({"ntl": ntl, "load": load}).dropna()
    n_possible = int(load.notna().sum())
    n_pairs = len(frame)
    r = np.nan
    if (
        n_pairs >= 3
        and frame.ntl.std(ddof=0) > 0
        and frame.load.std(ddof=0) > 0
    ):
        r = float(frame.ntl.corr(frame.load))
    direct_n = int((direct_mask & load.notna()).sum())
    return {
        "n_pairs": n_pairs,
        "N_pct": 100 * n_pairs / n_possible if n_possible else np.nan,
        "direct_n": direct_n,
        "direct_pct": 100 * direct_n / n_possible if n_possible else np.nan,
        "r": r,
        "score": (n_pairs / n_possible) * max(r, 0) ** 2 if n_possible and np.isfinite(r) else 0.0,
    }

def evaluate_region(region, windows=WINDOWS, sc_grid=SC_GRID):
    load = DATA_REGISTRY[region]["load"]
    rows = []
    for code in ORDERED_MASKS:
        data = DATA_REGISTRY[region]["masks"][code]
        for tau in sc_grid:
            direct_mask = data.ntl.notna() & data.SC.ge(tau)
            qualified = data.ntl.where(direct_mask)
            direct_counts = direct_mask.astype(int)
            for width in windows:
                ntl_mean = qualified.rolling(int(width), min_periods=1, center=True).mean()
                load_mean = load.rolling(int(width), min_periods=1, center=True).mean()
                count = direct_counts.rolling(int(width), min_periods=1, center=True).sum()
                metrics = paired_metrics(ntl_mean, load_mean, direct_mask, int(width))
                paired = pd.DataFrame({"ntl": ntl_mean, "load": load_mean, "count": count}).dropna()
                rows.append({
                    "region": region,
                    "mask": code,
                    "mask_label": GROUP_LOOKUP[code]["label"],
                    "tau": float(tau),
                    "window_days": int(width),
                    "mean_direct_days": float(paired["count"].mean()) if len(paired) else np.nan,
                    "mean_window_support_pct": (
                        100 * float(paired["count"].mean()) / width if len(paired) else np.nan
                    ),
                    **metrics,
                })
    return pd.DataFrame(rows)

SENS_PRIMARY = evaluate_region(PRIMARY_REGION)
SENS_PRIMARY.to_csv(TABLE_SUPP / "samar_leyte_mask_sc_window_sensitivity.csv.gz", index=False, compression="gzip")
print(f"{len(SENS_PRIMARY):,} primary-region configurations calculated")

In [ ]:
# Display grid retained from Notebook 3: τ = 5, 10, ..., 95%
DISPLAY_TAUS_5 = np.arange(5, 100, 5, dtype=float)


def mask_tau_matrix(
    frame,
    metric,
    window_days=1,
    region=PRIMARY_REGION,
    display_taus=DISPLAY_TAUS_5,
):
    subset = frame[
        frame.region.eq(region)
        & frame.window_days.eq(window_days)
    ]

    return (
        subset
        .pivot(index="mask", columns="tau", values=metric)
        .reindex(index=ORDERED_MASKS, columns=display_taus)
    )


# Retained because later supplementary figures use these helpers.
def heatmap_colour_values(metric, values):
    values = np.asarray(values, dtype=float)

    if metric == "r":
        return np.sign(values) * np.abs(values) ** R_COLOUR_POWER

    if metric == "N_pct":
        return (np.clip(values, 0, 100) / 100) ** N_COLOUR_POWER

    return values


def heatmap_style(metric, score_max=None):
    if metric == "r":
        return dict(
            colorscale=R_CONTRAST_SCALE,
            cmin=-1,
            cmax=1,
            ticks=R_COLOUR_TICKS,
            ticktext=[f"{value:g}" for value in R_TICKS],
            title="Pearson r",
        )

    if metric == "N_pct":
        return dict(
            colorscale=N_CONTRAST_SCALE,
            cmin=0,
            cmax=1,
            ticks=N_COLOUR_TICKS,
            ticktext=[f"{value:g}%" for value in N_TICKS],
            title="N%",
        )

    maximum = max(0.01, float(score_max or 1))

    return dict(
        colorscale=SCORE_CONTRAST_SCALE,
        cmin=0,
        cmax=maximum,
        ticks=None,
        ticktext=None,
        title="Score",
    )


def plot_daily_metric_heatmap(frame, metric):
    """
    Reproduces the style of Notebook 3, Figure 6.

    Panels:
        r       = Pearson correlation
        N_pct   = retained direct-date coverage
        score   = (N% / 100) × max(r, 0)^2
    """

    matrix = mask_tau_matrix(
        frame,
        metric,
        window_days=1,
        region=PRIMARY_REGION,
        display_taus=DISPLAY_TAUS_5,
    )

    z = matrix.to_numpy(dtype=float)

    y_labels = [
        f"{code}: {GROUP_LOOKUP[code]['label']}"
        for code in matrix.index
    ]

    if metric == "r":
        colorscale = "Blues"
        zmin, zmax = 0, 0.60
        highlight = z >= 0.60
        text_format = "{:.2f}"

        colorbar = dict(
            title=dict(text="Pearson r", side="right"),
            len=0.85,
            tickvals=[0, 0.20, 0.40, 0.60],
            ticktext=["0", "0.20", "0.40", "≥0.60"],
        )

        stem = "fig_06a_native_daily_r"

    elif metric == "N_pct":
        colorscale = "Greens"
        zmin, zmax = 0, 50
        highlight = z >= 50
        text_format = "{:.0f}"

        colorbar = dict(
            title=dict(text="N (%)", side="right"),
            len=0.85,
            tickvals=[0, 10, 20, 30, 40, 50],
            ticktext=["0", "10", "20", "30", "40", "≥50"],
        )

        stem = "fig_06b_native_daily_N"

    elif metric == "score":
        colorscale = "Pinkyl"
        zmin = 0
        zmax = max(0.01, float(np.nanmax(z)))

        finite_scores = z[np.isfinite(z)]
        top_decile = (
            np.nanpercentile(finite_scores, 90)
            if finite_scores.size
            else np.nan
        )

        highlight = z >= top_decile
        text_format = "{:.2f}"

        score_ticks = np.linspace(zmin, zmax, 5)

        colorbar = dict(
            title=dict(text="Composite score", side="right"),
            len=0.85,
            tickvals=score_ticks,
            ticktext=[f"{value:.2f}" for value in score_ticks],
        )

        stem = "fig_06c_native_daily_score"

    else:
        raise ValueError(
            "metric must be 'r', 'N_pct', or 'score'"
        )

    fig = go.Figure()

    fig.add_trace(
        go.Heatmap(
            z=z,
            x=matrix.columns,
            y=y_labels,
            colorscale=colorscale,
            zmin=zmin,
            zmax=zmax,
            xgap=2,
            ygap=2,
            colorbar=colorbar,
            showscale=True,
            hovertemplate=(
                "Mask %{y}<br>"
                "Spatial completeness τ=%{x:.0f}%<br>"
                "Value=%{z:.3f}"
                "<extra></extra>"
            ),
        )
    )

    # Annotate every cell; bold favourable configurations.
    for row in range(z.shape[0]):
        for col in range(z.shape[1]):
            value = z[row, col]

            if not np.isfinite(value):
                continue

            value_text = text_format.format(value)

            if highlight[row, col]:
                value_text = f"<b>{value_text}</b>"

            fig.add_annotation(
                x=matrix.columns[col],
                y=y_labels[row],
                text=value_text,
                showarrow=False,
                font=dict(
                    family="Arial",
                    size=14,
                    color="white" if highlight[row, col] else "black",
                ),
            )

    fig.update_layout(
        template="plotly_white",
        width=1500,
        height=400,
        margin=dict(
            l=175,
            r=45,
            t=20,
            b=80,
        ),
        xaxis=dict(
            title="Spatial completeness threshold, τ (%)",
            tickvals=list(range(0, 101, 20)),
            tickfont=dict(size=14),
        ),
        yaxis=dict(
            tickfont=dict(size=14),
        ),
        font=dict(
            family="Arial",
            color=INK,
            size=16,
        ),
        paper_bgcolor="rgba(0,0,0,0)",
    )

    return fig, stem


# ------------------------------------------------------------------
# Main Figure panels: r, direct-date N%, and the revised composite
# ------------------------------------------------------------------

fig_10a, stem_10a = plot_daily_metric_heatmap(
    SENS_PRIMARY,
    "r",
)

fig_10b, stem_10b = plot_daily_metric_heatmap(
    SENS_PRIMARY,
    "N_pct",
)

fig_10c, stem_10c = plot_daily_metric_heatmap(
    SENS_PRIMARY,
    "score",
)

fig_10a.show()
fig_10b.show()
fig_10c.show()

MAIN_DAILY_R_PANEL = export_figure(fig_10a, (FIG_MAIN) / (stem_10a), width=1500, height=400)

MAIN_DAILY_N_PANEL = export_figure(fig_10b, (FIG_MAIN) / (stem_10b), width=1500, height=400)

MAIN_DAILY_SCORE_PANEL = export_figure(fig_10c, (FIG_MAIN) / (stem_10c), width=1500, height=400)


# ------------------------------------------------------------------
# Highest-scoring Samar–Leyte configurations on the 5% τ grid
# ------------------------------------------------------------------

DAILY_COMPONENT_ALL = (
    SENS_PRIMARY[
        SENS_PRIMARY.window_days.eq(1)
    ][
        [
            "mask",
            "mask_label",
            "tau",
            "r",
            "n_pairs",
            "N_pct",
            "direct_pct",
            "score",
        ]
    ]
    .copy()
)

DAILY_COMPONENT_5PCT = (
    DAILY_COMPONENT_ALL[
        DAILY_COMPONENT_ALL.tau.isin(DISPLAY_TAUS_5)
    ]
    .sort_values(
        ["score", "N_pct", "r", "tau", "mask"],
        ascending=[False, False, False, True, True],
    )
    .reset_index(drop=True)
)

TOP_SAMAR_LEYTE_5PCT = (
    DAILY_COMPONENT_5PCT
    .head(10)
    .copy()
)

TOP_SAMAR_LEYTE_5PCT.insert(
    0,
    "rank",
    np.arange(1, len(TOP_SAMAR_LEYTE_5PCT) + 1),
)

display(
    Markdown(
        "### Highest-scoring native-daily Samar–Leyte "
        "configurations (5% τ grid)"
    )
)

display(
    TOP_SAMAR_LEYTE_5PCT.round(
        {
            "tau": 0,
            "r": 3,
            "N_pct": 1,
            "direct_pct": 1,
            "score": 4,
        }
    )
)

TOP_SAMAR_LEYTE_5PCT.to_csv(
    TABLE_MAIN
    / "section1_top10_samar_leyte_daily_5pct_tau.csv",
    index=False,
)

DAILY_COMPONENT_5PCT.to_csv(
    TABLE_MAIN
    / "section1_samar_leyte_daily_5pct_tau_all.csv",
    index=False,
)

# Retain the full 1% results for audit and sensitivity reporting.
DAILY_COMPONENT_ALL.to_csv(
    TABLE_SUPP
    / "section1_samar_leyte_daily_exact_1pct_tau_all.csv",
    index=False,
)

## 8. Figure 7 — Samar–Leyte time series and Haiyan zoom

The layout shows the eight cumulative masks as separate rows, with native-daily points and the 30-day reference curves. Its plot-specific preparation uses upper-tail exclusion, a strict SC cutoff and NGCP screening. The panel annotations come from those plotted series, while the selection sweep above uses P95 capping and an inclusive cutoff.

In [ ]:
# ----------------------------------------------------------
# Project and input paths
# ----------------------------------------------------------
PROJECT_ROOT = ROOT

DATE_START = pd.Timestamp(globals().get("DATE_START", "2012-01-19"))
DATE_END = pd.Timestamp(globals().get("DATE_END", "2024-12-31"))
ANALYSIS_DAYS = (DATE_END - DATE_START).days + 1
VALUE_COL = globals().get("VALUE_COL", "Mean_by_sum")

# Preserve metadata/helpers already defined in the notebook.
if "GHSL_META" not in globals():
    GHSL_META = {
        "30": {"code": "G1", "label": "Urban Core", "color": "#ff0000"},
        "23-30": {"code": "G2", "label": "Dense Urban", "color": "#732600"},
        "22-30": {"code": "G3", "label": "Core & Inner", "color": "#a87000"},
        "21-30": {"code": "G4", "label": "Urban Fringe", "color": "#ffff00"},
        "13-30": {"code": "G5", "label": "Built-Up / Rural Mix", "color": "#375623"},
        "12-30": {"code": "G6", "label": "Low–Mid Rural", "color": "#abcd66"},
        "11-30": {"code": "G7", "label": "Rural Settlements", "color": "#cdf57a"},
        "10-30": {"code": "G8", "label": "All Land", "color": "#7ab6f5"},
    }

if "GHSL_ORDER" not in globals():
    GHSL_ORDER = [
        "30", "23-30", "22-30", "21-30",
        "13-30", "12-30", "11-30", "10-30",
    ]

if "STORM_MARKERS" not in globals():
    STORM_MARKERS = [
        ("Haiyan", "2013-11-08"),
        ("Rai", "2021-12-16"),
    ]

if "zscore" not in globals():
    def zscore(series):
        s = pd.Series(series).astype(float)
        sd = s.std(ddof=0)
        if not np.isfinite(sd) or sd == 0:
            return pd.Series(np.nan, index=s.index)
        return (s - s.mean()) / sd

df_all = NTL_PANEL

In [ ]:
def build_alignment_source(df_all, ngcp_df, region, tau):
    dates = pd.date_range(DATE_START, DATE_END, freq="D")

    ng = pd.to_numeric(ngcp_df[region], errors="coerce").copy()
    ng.index = pd.to_datetime(ng.index).normalize()
    if ng.index.has_duplicates:
        raise ValueError(f"Duplicate NGCP dates: {region}")
    ng = ng.sort_index().reindex(dates)

    # NGCP screening, applied once to the original input.
    window = ng.rolling(31, center=True, min_periods=7)
    median = window.median()
    mad = window.apply(
        lambda x: np.nanmedian(np.abs(x - np.nanmedian(x))),
        raw=True,
    )
    sigma = 1.4826022185 * mad
    flagged = (sigma > 0) & ((ng - median).abs() > 3 * sigma)
    ng = ng.mask(flagged)

    ng_daily = zscore(ng)
    ng_30d = ng_daily.rolling("30D", min_periods=1).mean()

    # Preserve the trace positions expected by your final layout cell.
    source = make_subplots(rows=2 * len(GHSL_ORDER), cols=2)
    all_values = [ng_daily.to_numpy(), ng_30d.to_numpy()]

    for i, group in enumerate(GHSL_ORDER):
        gdf = df_all[
            (df_all["Region"] == region)
            & (df_all["Group"] == group)
        ].copy()
        gdf = gdf[gdf["Valid_px"] > 0].copy()
        gdf["date"] = pd.to_datetime(gdf["date"]).dt.normalize()
        gdf = gdf.sort_values("date")

        if gdf.empty:
            raise ValueError(f"No NTL data: {region}, {group}")

        hi = np.nanpercentile(gdf[VALUE_COL], 95)
        gdf = gdf[gdf[VALUE_COL] <= hi].copy()

        if gdf.empty:
            raise ValueError(f"No NTL data after filtering: {group}")
        if gdf["date"].duplicated().any():
            raise ValueError(f"Duplicate NTL dates: {group}")

        gdf["SC_pct"] = gdf["Valid_px"] / gdf["Valid_px"].max() * 100
        gdf = gdf.set_index("date").reindex(dates)
        ntl = gdf[VALUE_COL].where(gdf["SC_pct"] > tau)

        paired = ntl.notna() & ng.notna()
        reference = ntl.loc[paired]
        sd = reference.std(ddof=0)

        if np.isfinite(sd) and sd > 0:
            ntl_scaled = (ntl - reference.mean()) / sd
        else:
            ntl_scaled = pd.Series(np.nan, index=dates)

        ntl_daily = ntl_scaled.where(paired)
        ntl_30d = ntl_scaled.rolling("30D", min_periods=1).mean()

        all_values.extend([
            ntl_daily.to_numpy(),
            ntl_30d.to_numpy(),
        ])

        # Order required by the final layout:
        # NGCP daily, NTL daily, NTL 30-day, NGCP 30-day.
        for series, mode, color in [
            (ng_daily, "markers", "#666666"),
            (ntl_daily, "markers", GHSL_META[group]["color"]),
            (ntl_30d, "lines", GHSL_META[group]["color"]),
            (ng_30d, "lines", "black"),
        ]:
            source.add_trace(
                go.Scatter(
                    x=dates.strftime("%Y-%m-%d").tolist(),
                    y=series.to_numpy(),
                    mode=mode,
                    marker=dict(color=color),
                    line=dict(color=color),
                    connectgaps=False,
                    showlegend=False,
                ),
                row=2 * i + 2,
                col=1,
            )

    values = np.concatenate(all_values)
    values = values[np.isfinite(values)]
    if not values.size:
        raise ValueError("No finite observations remain.")

    padding = max(0.04 * (values.max() - values.min()), 0.1)

    source.update_xaxes(
        type="date",
        range=[
            dates.min().strftime("%Y-%m-%d"),
            dates.max().strftime("%Y-%m-%d"),
        ],
        row=2, col=1,
    )
    source.update_yaxes(
        range=[values.min() - padding, values.max() + padding],
        row=2, col=1,
    )
    return source


REGION = "Samar-Leyte"
TAU = 25

fig_alignment_all8 = build_alignment_source(
    df_all,
    ngcp_df,
    region=REGION,
    tau=TAU,
)

In [ ]:
source = fig_alignment_all8
groups = list(GHSL_ORDER)

haiyan = pd.Timestamp("2013-11-08")
zoom_start = haiyan - pd.Timedelta(days=60)
zoom_end = haiyan + pd.Timedelta(days=180)

full_start, full_end = pd.to_datetime(source.layout.xaxis3.range)
views = [(full_start, full_end), (zoom_start, zoom_end)]

HEIGHT = 2100
TOP = 75
BOTTOM = 85
inner_height = HEIGHT - TOP - BOTTOM

fig = make_subplots(
    rows=len(groups),
    cols=2,
    shared_xaxes="columns",
    column_widths=[0.75, 0.25],
    horizontal_spacing=0.045,
    vertical_spacing=0.018,
)

old_limits = source.layout.yaxis3.range
y_limits = [
    min(float(old_limits[0]), -2.2),
    max(float(old_limits[1]), 2.2),
]


def correlation_and_count(ntl_trace, ngcp_trace):
    ntl = pd.Series(
        np.asarray(ntl_trace.y, dtype=float),
        index=pd.to_datetime(ntl_trace.x),
        name="NTL",
    )
    load = pd.Series(
        np.asarray(ngcp_trace.y, dtype=float),
        index=pd.to_datetime(ngcp_trace.x),
        name="NGCP",
    )
    paired = pd.concat([ntl, load], axis=1).dropna()
    r = paired["NTL"].corr(paired["NGCP"]) if len(paired) >= 2 else np.nan
    return r, len(paired)


for i, group in enumerate(groups):
    row = i + 1
    meta = GHSL_META[group]

    # Always take the full-record data, including dates beyond
    # the previous +120-day zoom.
    original_traces = [
        t for t in source.data
        if t.type == "scatter"
        and t.xaxis == f"x{4 * i + 3}"
        and not t.showlegend
    ]

    base_bottom, base_top = fig.get_subplot(row, 1).yaxis.domain
    plot_top = base_top - 38 / inner_height

    for col, (start, end) in enumerate(views, start=1):
        panel_traces = []

        for j, trace in enumerate(original_traces):
            dates = pd.DatetimeIndex(pd.to_datetime(trace.x))
            selected = (dates >= start) & (dates <= end)

            copied = go.Scatter(trace.to_plotly_json())
            copied.x = dates[selected].strftime("%Y-%m-%d").tolist()
            copied.y = np.asarray(trace.y, dtype=float)[selected]
            copied.showlegend = False

            is_ngcp = j in (0, 3)

            if copied.mode == "markers":
                copied.marker.size = 3.2 if col == 1 else 4.8
                copied.marker.opacity = (
                    (0.13 if col == 1 else 0.22)
                    if is_ngcp else
                    (0.24 if col == 1 else 0.38)
                )
            else:
                copied.line.width = (
                    (1.9 if col == 1 else 2.5)
                    if is_ngcp else
                    (1.7 if col == 1 else 2.2)
                )
                copied.connectgaps = False

            panel_traces.append(copied)
            fig.add_trace(copied, row=row, col=col)

        # Trace order: NGCP daily, NTL daily, NTL mean, NGCP mean.
        ng_daily, ntl_daily, ntl_30d, ng_30d = panel_traces
        r1, count = correlation_and_count(ntl_daily, ng_daily)
        r30, _ = correlation_and_count(ntl_30d, ng_30d)

        possible_days = (
            ANALYSIS_DAYS if col == 1 else (end - start).days + 1
        )
        coverage = 100 * count / possible_days

        r1_text = f"{r1:.2f}" if np.isfinite(r1) else "—"
        r30_text = f"{r30:.2f}" if np.isfinite(r30) else "—"

        label = (
            f"{meta['code']}: {meta['label']}"
            if col == 1 else meta["code"]
        )
        dot = f"<span style='color:{meta['color']}'>●</span>"
        line = f"<span style='color:{meta['color']}'>━</span>"

        # One text object, no title and no line breaks.
        key = (
            f"{label} {dot} r₁={r1_text}  "
            f"{line} r₃₀={r30_text}  "
            f"N={coverage:.0f}%  n={count}"
        )

        x_domain = fig.get_subplot(row, col).xaxis.domain
        fig.add_annotation(
            x=x_domain[1],
            y=base_top,
            xref="paper",
            yref="paper",
            text=key,
            xanchor="right",
            yanchor="top",
            showarrow=False,
            font=dict(
                family="Arial",
                size=18 if col == 1 else 17,
                color="#333333",
            ),
        )

        fig.update_yaxes(
            domain=[base_bottom, plot_top],
            range=y_limits,
            tickmode="array",
            tickvals=[-2, 0, 2],
            ticktext=["−2", "0", "+2"],
            tickfont=dict(size=17),
            showticklabels=True,
            zeroline=True,
            zerolinecolor="#dddddd",
            row=row, col=col,
        )
        fig.update_xaxes(
            type="date",
            range=[
                start.strftime("%Y-%m-%d"),
                end.strftime("%Y-%m-%d"),
            ],
            showticklabels=False,
            showgrid=False,
            row=row, col=col,
        )

        if col == 1:
            fig.update_yaxes(
                title_text="z-score",
                title_font=dict(size=19),
                row=row, col=col,
            )
            fig.add_vrect(
                x0=zoom_start.strftime("%Y-%m-%d"),
                x1=zoom_end.strftime("%Y-%m-%d"),
                line_color="#555555",
                line_width=1.5,
                fillcolor="rgba(0,0,0,0)",
                layer="above",
                row=row, col=col,
            )
        else:
            for update_axis in (fig.update_xaxes, fig.update_yaxes):
                update_axis(
                    showline=True,
                    mirror=True,
                    linecolor="#555555",
                    linewidth=1.5,
                    row=row, col=col,
                )

        for _, event_date in STORM_MARKERS:
            if start <= pd.Timestamp(event_date) <= end:
                fig.add_vline(
                    x=event_date,
                    line_color="black",
                    line_dash="dash",
                    line_width=0.9,
                    opacity=0.5,
                    row=row, col=col,
                )

# Shared NGCP key: one text object guarantees one horizontal line.
fig.add_annotation(
    x=0.6,
    y=0.92,
    xref="paper",
    yref="paper",
    text=(
        f"NGCP Electric Load<br>"
        "<span style='color:#666666'>●</span> W=1 "
        "<span style='color:black'>━</span> W=30"
    ),
    xanchor="left",
    yanchor="top",
    align="left",
    showarrow=False,
    font=dict(family="Arial", size=16, color="#333333"),
)
fig.add_annotation(
    x=0.72,
    y=0.975,
    xref="paper",
    yref="paper",
    text=f"(τ = {TAU}%)",
    xanchor="right",
    yanchor="bottom",
    showarrow=False,
    font=dict(size=19),
)

fig.update_xaxes(
    showticklabels=True,
    title_text="Date",
    title_font=dict(size=19),
    tickformat="%Y",
    dtick="M48",
    tickfont=dict(size=17),
    tickangle=0,
    row=len(groups), col=1,
)
fig.update_xaxes(
    showticklabels=True,
    title_text="Date",
    title_font=dict(size=19),
    tickformat="%b %Y",
    dtick="M3",
    tickfont=dict(size=17),
    tickangle=0,
    row=len(groups), col=2,
)

fig.update_layout(
    template="plotly_white",
    width=1550,
    height=HEIGHT,
    margin=dict(t=TOP, b=BOTTOM, l=95, r=45),
    font=dict(family="Arial", size=18, color="#333333"),
    paper_bgcolor="rgba(0,0,0,0)",
    showlegend=False,
)

fig_alignment_paper_final = fig
fig_alignment_paper_final.show()

In [ ]:
export_figure(fig_alignment_paper_final, (OUTPUT_MAIN_FIG_DIR) / ("fig_07_samar_leyte"), width=fig_alignment_paper_final.layout.width, height=fig_alignment_paper_final.layout.height, scale=3)

## 9. Outputs

Figures 6a–c and 7 are exported once to outputs/figures/main/. Supplementary figures use outputs/figures/supplementary/, and tables use outputs/tables/.

## Conclusion

Configuration assessment requires alignment and observation support together. The heatmaps show their trade-off, while the time-series panels retain event and full-record context. Notebook 14 repeats the daily assessment across sub-grids and Notebook 15 evaluates sensitivity.